In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are now expected on the local filesystem. Set LOCAL_DATA_DIR
# (or rely on the notebook's working directory) to point to the folder that
# contains data.csv, predict.csv, etc.

import os
from pathlib import Path

LOCAL_DATA_DIR = Path(os.environ.get("LOCAL_DATA_DIR", Path.cwd())).resolve()
print(f"LOCAL_DATA_DIR set to {LOCAL_DATA_DIR}")

In [ ]:
## Starter Code for Building Baseball Win Prediction Model

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Load the pre-processed datasets from the directory pointed to by LOCAL_DATA_DIR
DATA_DIR = LOCAL_DATA_DIR  # defined in the previous cell
print(f"Loading datasets from {DATA_DIR}")

data_df = pd.read_csv(DATA_DIR / "data.csv")N
predict_df = pd.read_csv(DATA_DIR / "predict.csv")

# Display basic information about the datasets
print(f"Data set shape: {data_df.shape}")
print(f"Predict set shape: {predict_df.shape}")

In [ ]:
# Use every numeric/boolean column shared by data.csv and predict.csv
# (drops the target, identifiers and text labels automatically)
exclude_cols = {'W', 'ID', 'yearID', 'teamID', 'year_label', 'decade_label', 'win_bins'}

shared_cols = [col for col in data_df.columns if col in predict_df.columns]
available_features = [
    col for col in data_df[shared_cols].select_dtypes(include=['number', 'bool']).columns
    if col not in exclude_cols
]
print(f"Number of available features: {len(available_features)}")

# Separate features and target variable
X = data_df[available_features]
y = data_df['W']

# Perform the split (adjust test_size / random_state as needed)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% for testing
    random_state=42    # ensures reproducibility
)

In [ ]:
# Scale features
# Identify columns to exclude from scaling (one-hot encoded and label columns)
one_hot_cols = [col for col in X_train.columns if col.startswith(('era_', 'decade_'))]
other_cols = [col for col in X_train.columns if col not in one_hot_cols]

# Scale only non-one-hot features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[other_cols] = scaler.fit_transform(X_train[other_cols])
X_test_scaled[other_cols] = scaler.transform(X_test[other_cols])

In [ ]:
# Build Linear Regression Model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_train_preds = lr.predict(X_train_scaled)
lr_test_preds = lr.predict(X_test_scaled)

In [ ]:
# Evaluate Linear Regression
lr_train_mae = mean_absolute_error(y_train, lr_train_preds)
lr_test_mae = mean_absolute_error(y_test, lr_test_preds)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_preds))
lr_test_r2 = r2_score(y_test, lr_test_preds)

print(f"Linear Regression Performance:")
print(f"  Training MAE: {lr_train_mae:.4f}")
print(f"  Test MAE: {lr_test_mae:.4f}")
print(f"  Test RMSE: {lr_test_rmse:.4f}")
print(f"  Test R²: {lr_test_r2:.4f}")

# Feature importance from Linear Regression
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

In [ ]:
# Visualize actual vs predicted values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, lr_test_preds, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Wins')
plt.ylabel('Predicted Wins')
plt.title('Linear Regression: Actual vs Predicted Wins')
plt.grid(True, alpha=0.3)

# Add residual plot
plt.figure(figsize=(10, 6))
residuals = y_test - lr_test_preds
plt.scatter(lr_test_preds, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Wins')
plt.ylabel('Residuals')
plt.title('Linear Regression: Residual Plot')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Prepare Submission

# Align features and scale using the previously fitted scaler
predict_scaled = predict_df[available_features].copy()
predict_scaled[other_cols] = scaler.transform(predict_scaled[other_cols])

# Predict wins for the new dataset (replace `lr` with your final model if you change it)
predict_preds = lr.predict(predict_scaled)

# Build submission in the same format as submission.csv
submission_df = pd.DataFrame({
    'ID': predict_df['ID'],
    'W': np.round(predict_preds).astype(int)
})

submission_path = 'sample_submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f"Kaggle submission saved to {submission_path}")

## Next steps: how to improve your model

You now have a working baseline (linear regression on the raw team stats). Use the three areas below as starting points to explore on your own. Judge every change with the **same validation approach** (see the tuning section) so you know whether it really helped.

### 1. Feature engineering

The raw columns are only a starting point. Think about what actually makes a baseball team win, and whether you can turn the existing columns into new ones that describe it better.


- **Adjust for context:** seasons differ in the number of games played, and the amount of scoring has changed over baseball's history. For example, the average team scored about 3.4 runs per game in 1908 but about 5.5 in 1930, so 700 runs would be an excellent season in one era and an average one in the other. How could you adjust the stats so teams from different seasons can be compared fairly?
- **Combine columns:** differences, ratios or totals of existing stats can say more than each one on its own. For example, a ratio like hits divided by at-bats (`H / AB`) tells you how often a team gets a hit, not just how many it got. Two teams can both record 1,400 hits, but if one needed far more at-bats to get there, it was less effective at the plate. The ratio puts the raw count in the context of the opportunities behind it.
- **Look beyond straight lines:** does the relationship between a stat and wins always look the same?
- **Remove or reduce:** are some columns telling you the same thing? Would fewer features work just as well?


The feature-selection cell above uses every numeric column shared by both files, so any new column needs to be added to **both** `data_df` and `predict_df` before that cell runs.


### 2. Try different ML models

Swap the model while keeping the same features and the same train/test split, then compare MAE, RMSE and R² (ideally using the cross-validation approach described in the tuning section below). This list is **not exhaustive**: treat it as a set of starting points, and feel free to try other models.

- **Regularised linear models:** `Ridge`, `Lasso`, `ElasticNet`. Plain linear regression chooses whatever coefficients fit the training data best, which can make them large and unstable when features overlap or there are many of them. Regularised versions add a penalty on large coefficients, accepting a slightly worse fit on the training data in return for a simpler model that often generalises better. `Ridge` shrinks all coefficients, `Lasso` can shrink some to exactly zero (effectively dropping those features), and `ElasticNet` mixes the two.
- **Tree ensembles:** `RandomForestRegressor`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`. Instead of one formula, these combine many decision trees (series of yes/no splits on the features). They can pick up curved relationships and interactions between features without you creating them. A random forest builds many trees independently and averages them, while gradient boosting builds trees one after another, each trying to fix the mistakes of the ones before.
- **Gradient boosting libraries:** `XGBoost`, `LightGBM`. Faster, more heavily optimised versions of the boosting idea above, with many more settings to tune. They are popular in data science competitions.
- **Other families:** `SVR` fits a flexible curve while tolerating small errors, and `KNeighborsRegressor` predicts by averaging the most similar teams in the training data. Both are sensitive to the scale of the features, so use the scaled data from earlier (tree-based models do not need it).
- **Stacking:** combine several different models, since their mistakes often partly cancel out. `StackingRegressor` trains another model (the "final estimator") to learn the best way to combine the predictions of the models you give it. `VotingRegressor` is a simpler alternative that just averages their predictions.

Things to look out for: compare **training vs test error** to spot overfitting (a large gap means the model is memorising), and remember that a more complex model is not automatically better. Check whether it beats the linear baseline by enough to matter.



### 3. Hyperparameter tuning

Hyperparameters (settings you choose before training, such as tree depth or regularisation strength) can change performance.

- **Use cross-validation, not one split:** `KFold` / `cross_val_score` give a more reliable estimate than a single `train_test_split`, which can be lucky or unlucky. Wrap the scaler and model in a `Pipeline` so scaling is refit inside each fold.
- **Grid search:** `GridSearchCV` tries every combination you list (good for small search spaces).
- **Random search:** `RandomizedSearchCV` samples combinations (more efficient for larger spaces).
- **What to tune** (examples only, not an exhaustive list):
  - Ridge / Lasso: `alpha`
  - Random Forest: `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`
  - XGBoost / LightGBM: `learning_rate`, `n_estimators`, `max_depth`, `subsample`, `colsample_bytree`, `reg_lambda`

A good workflow: set a baseline, change **one thing at a time**, record the cross-validated score, and keep only the changes that clearly help.